# LAB 4c: Keras Wide & Deep モデルの作成

**学習目標**

1. CSV カラム、ラベルカラム、およびカラムのデフォルト値を設定する
1. CSV ファイルから特徴量とラベルのデータセットを作成する
1. 生の特徴量に対する入力レイヤーを作成する
1. 生の特徴量を変換する前処理レイヤーを定義する
1. Wide レイヤー、Deep の全結合（Dense）隠れレイヤー、および出力レイヤーを作成する
1. カスタム評価指標を作成する
1. すべてのコンポーネントを結合して Wide & Deep モデルを構築する
1. モデルの訓練と評価を行う


## はじめに
このノートブックでは、Keras を使用して Wide & Deep モデルを作成し、赤ちゃんが生まれる前の出生時体重を予測します。

まず、データ入力用の CSV カラム名、ラベルカラム、およびカラムのデフォルト値を定義します。次に、CSV ファイルから特徴量とラベルで構成される `tf.data Dataset` を構築し、生の特徴量に対する入力レイヤーを作成します。続いて、モデル入力用の前処理レイヤーをセットアップし、Keras で Wide & Deep ニューラルネットワークを構築します。カスタム評価指標を作成して Wide & Deep モデルを構築し、最後にモデルの訓練と評価を行います。


## 必要なライブラリの読み込み


In [ ]:
import datetime
import os
import shutil

import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

## CSV ファイルが存在することを確認する

このシリーズの 7 つ目のラボ [4a_sample_babyweight](../solutions/4a_sample_babyweight.ipynb) では、BigQuery からサンプリングを行って訓練、評価、テスト用の CSV ファイルを作成しました。これらが存在することを確認してください。存在しない場合は、そのラボに戻って作成してください。


In [ ]:
%%bash
ls *.csv

In [ ]:
%%bash
head -5 *.csv

## Keras モデルの作成


### CSV カラム、ラベルカラム、およびカラムのデフォルト値の設定

CSV ファイルが存在することを確認できたので、入力関数で使用するいくつかの設定を行います。
* `CSV_COLUMNS`: カラムのヘッダー名。CSV ファイルと同じ順序になっていることを確認してください。
* `LABEL_COLUMN`: ラベルとなるカラムのヘッダー名。特徴量の辞書から取り出す（pop する）ために必要になります。
* `DEFAULTS`: `CSV_COLUMNS` と同じ長さのリスト。つまり、CSV の各カラムに対するデフォルト値です。各要素自体が、その CSV カラムのデフォルト値を含むリストです。


In [ ]:
# CSV、ラベル、およびキーカラムを決定
CSV_COLUMNS = [
    "weight_pounds",
    "is_male",
    "mother_age",
    "plurality",
    "gestation_weeks",
]
LABEL_COLUMN = "weight_pounds"

NUMERICAL_COLUMNS = ["mother_age", "gestation_weeks"]
CATEGORICAL_COLUMNS = ["is_male", "plurality"]

# 各 CSV カラムのデフォルト値を設定
# is_male と plurality は文字列として扱います
DEFAULTS = [[0.0], ["null"], [0.0], ["null"], [0.0]]

### CSV ファイルからの特徴量とラベルのデータセット作成

次に、データを読み込む `input_fn` を作成します。これにより CSV データセットオブジェクトが作成されます。ただし、カラムを特徴量とラベルに分割する必要があります。これは、データセットに map メソッドを適用し、特徴量テンソルの辞書からラベルカラムを取り出す（pop する）ことで行えます。


In [ ]:
def features_and_labels(row_data):
    """特徴量辞書から特徴量とラベルを分離します。

    Args:
        row_data: CSV カラム名とテンソル値の辞書。
    Returns:
        特徴量テンソルの辞書とラベルテンソル。
    """
    label = row_data.pop(LABEL_COLUMN)

    return row_data, label


def decode_csv(line):
    fields = tf.io.decode_csv(line, record_defaults=DEFAULTS)
    return dict(zip(CSV_COLUMNS, fields))


def load_dataset(pattern, batch_size=1, mode="eval"):
    """tf.data API を使用して CSV ファイルからデータセットを読み込みます。

    Args:
        pattern: str, ファイル一覧に展開するファイルパターン。
        batch_size: int, 1 バッチあたりのサンプル数。
        mode: 訓練（train）か評価（eval）かを指定。
    Returns:
        `Dataset` オブジェクト。
    """
    dataset = tf.data.Dataset.list_files(pattern, shuffle=(mode == "train"))

    # これらのファイルの内容を読み込むために TextLineDataset を使用
    # interleave により複数のファイルから並列に読み込み可能
    dataset = dataset.interleave(
        tf.data.TextLineDataset, num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.map(decode_csv, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.map(
        map_func=features_and_labels, num_parallel_calls=tf.data.AUTOTUNE
    )
    dataset = dataset.batch(batch_size)
    # 訓練用にシャッフル
    if mode == "train":
        dataset = dataset.shuffle(buffer_size=1000)

    dataset = dataset.repeat().prefetch(buffer_size=tf.data.AUTOTUNE)

    return dataset

### 生の特徴量に対する入力レイヤーの作成

入力関数によって読み込まれたデータをモデル関数に渡す必要がありますが、どのように連携させればよいでしょうか？ Keras の入力レイヤー `keras.layers.Input` を使用して、以下を定義します:
* shape: バッチサイズを含まない形状タプル（整数）。例えば、shape=(32,) は想定される入力が 32 次元のベクトルのバッチであることを示します。このタプルの要素は None にすることもできます。「None」要素は、形状が不明な次元を表します。
* name: レイヤーの任意の名前文字列。モデル内で一意である必要があります（同じ名前を 2 回使用しないでください）。指定しない場合は自動生成されます。
* dtype: 入力で想定されるデータ型を文字列で指定（float32, float64, int32 など）。


In [ ]:
def create_input_layers():
    """各特徴量の入力レイヤーの辞書を作成します。

    Returns:
        各特徴量の `keras.Input` レイヤーの辞書。
    """
    inputs = {
        colname: keras.Input(name=colname, shape=(1,), dtype="float32")
        for colname in NUMERICAL_COLUMNS
    }

    inputs.update(
        {
            colname: keras.Input(name=colname, shape=(1,), dtype="string")
            for colname in CATEGORICAL_COLUMNS
        }
    )

    return inputs

### 入力用の前処理レイヤーの定義

次に、前処理レイヤーを定義します。`mother_age` と `gestation_weeks` は数値型である必要があります。その他の `is_male` と `plurality` はカテゴリ型である必要があります。


In [ ]:
def transform(inputs, nembeds):
    """変換された入力の辞書を作成します。

    Returns:
        変換されたテンソルの辞書。
    """

    deep = {}
    wide = {}

    buckets = {
        "mother_age": np.arange(15, 45, 1).tolist(),
        "gestation_weeks": np.arange(17, 47, 1).tolist(),
    }
    bucketized = {}

    for numeric_column in NUMERICAL_COLUMNS:
        deep[numeric_column] = inputs[numeric_column]
        bucketized[numeric_column] = keras.layers.Discretization(
            buckets[numeric_column]
        )(inputs[numeric_column])
        wide[f"btk_{numeric_column}"] = keras.layers.CategoryEncoding(
            num_tokens=len(buckets[numeric_column]) + 1, output_mode="one_hot"
        )(bucketized[numeric_column])

    crossed = keras.layers.HashedCrossing(
        num_bins=len(buckets["mother_age"]) * len(buckets["gestation_weeks"])
    )((bucketized["mother_age"], bucketized["gestation_weeks"]))

    deep["age_gestation_embeds"] = keras.layers.Flatten()(
        keras.layers.Embedding(
            input_dim=len(buckets["mother_age"])
            * len(buckets["gestation_weeks"]),
            output_dim=nembeds,
        )(crossed)
    )

    vocab = {
        "is_male": ["True", "False", "Unknown"],
        "plurality": [
            "Single(1)",
            "Twins(2)",
            "Triplets(3)",
            "Quadruplets(4)",
            "Quintuplets(5)",
            "Multiple(2+)",
        ],
    }

    for categorical_column in CATEGORICAL_COLUMNS:
        wide[categorical_column] = keras.layers.StringLookup(
            vocabulary=vocab[categorical_column], output_mode="one_hot"
        )(inputs[categorical_column])

    return wide, deep

### Wide & Deep モデルと出力レイヤーの作成

機械学習用に入力を準備する方法がわかりましたが、次に入力と期待する出力を結びつける必要があります。この 2 つを繋ぐのがモデルアーキテクチャです。ここで Wide & Deep モデルを作成します。Wide 側は単なる線形回帰または Dense レイヤーになります。Deep 側には、いくつかの全結合（Dense）隠れレイヤーを作成します。これらはすべて 1 つの Dense 出力レイヤーに統合されます。これは回帰問題なので、出力レイヤーの活性化関数と形状が正しいことを確認してください。


In [ ]:
def get_model_outputs(wide_inputs, deep_inputs, dnn_hidden_units):
    """モデルアーキテクチャを作成し、出力を返します。

    Args:
        wide_inputs: モデルの Wide 側の入力として使用される Dense テンソル。
        deep_inputs: モデルの Deep 側の入力として使用される Dense テンソル。
        dnn_hidden_units: 整数リスト。長さは隠れ層の数を表し、
            i 番目の要素は i 番目の層のニューロン数を表します。
    Returns:
        モデルからの Dense テンソル出力。
    """
    # Deep 側の隠れ層
    layers = [int(x) for x in dnn_hidden_units]
    deep = deep_inputs
    for layerno, numnodes in enumerate(layers):
        deep = keras.layers.Dense(
            units=numnodes, activation="relu", name=f"dnn_{layerno + 1}"
        )(deep)
    deep_out = deep

    # Wide 側の線形モデル
    wide_out = keras.layers.Dense(units=10, activation="relu", name="linear")(
        wide_inputs
    )

    # 2 つの側を結合（連結）
    both = keras.layers.Concatenate(name="both")([deep_out, wide_out])

    # 回帰問題であるため最終出力は線形活性化
    output = keras.layers.Dense(units=1, activation="linear", name="weight")(
        both
    )

    return output

### すべてのパーツを結合して Wide & Deep モデルを構築する

素晴らしい！ すべてのパーツが揃ったので、これらを 1 つの Keras Model にまとめます。これは分岐やサイドアームなどのない単純な順伝播モデルではないため、Keras の Sequential Model API は使用できません。代わりに、Keras の Functional Model API を使用します。ここでは、入力と出力を指定して `keras.models.Model` を使用してモデルを構築し、オプティマイザ、損失関数、評価指標を指定してモデルをコンパイルします。


In [ ]:
def build_wide_deep_model(dnn_hidden_units=[64, 32], nembeds=3):
    """Keras Functional API を使用して Wide & Deep モデルを構築します。

    Returns:
        `keras.models.Model` オブジェクト。
    """
    # 入力レイヤーを作成
    inputs = create_input_layers()

    # Wide と Deep の両方に対して生の特徴量を変換
    wide, deep = transform(inputs, nembeds)

    # Keras の Functional API では LayerConstructor()(inputs) の形式が必要
    wide_inputs = keras.layers.Concatenate()(list(wide.values()))
    deep_inputs = keras.layers.Concatenate()(list(deep.values()))

    # 入力に対するモデルの出力を取得
    output = get_model_outputs(wide_inputs, deep_inputs, dnn_hidden_units)

    # モデルを構築し、すべてまとめてコンパイル
    model = keras.models.Model(inputs=inputs, outputs=output)
    rmse = keras.metrics.RootMeanSquaredError(
        name="root_mean_squared_error", dtype=None
    )
    model.compile(optimizer="adam", loss="mse", metrics=[rmse, "mse"])

    return model


print("Here is our wide and deep architecture so far:\n")
model = build_wide_deep_model()
print(model.summary())

Keras の plot_model ユーティリティを使用して、Wide & Deep ネットワークを可視化できます。


In [ ]:
keras.utils.plot_model(model=model, show_layer_names=True, rankdir="LR")

## モデルの実行と評価


### 訓練と評価

CSV ファイルからの入力と設計したアーキテクチャを使用して Keras モデルを構築しました。次に、モデルパラメータを訓練し、訓練の進行に合わせて定期的に評価を実行して未知のデータに対するパフォーマンスを追跡することで、モデルを実行しましょう。訓練用と評価用の両方のデータセットを読み込み、fit メソッドを通じてモデルに渡す必要があります。データを読み込む際は、正しいファイルパターン、バッチサイズ、モードが設定されていることを確認してください。また、TensorBoard へのコールバックを追加することも忘れないでください。


In [ ]:
TRAIN_BATCH_SIZE = 32
NUM_TRAIN_EXAMPLES = 10000 * 5  # 訓練データセットはリピートされ、周回します
NUM_EVALS = 5  # 評価を行う回数
# 妥当なサンプル数を確保しつつ、処理が遅くならない程度の件数
NUM_EVAL_EXAMPLES = 10000

trainds = load_dataset(
    pattern="train*",
    batch_size=TRAIN_BATCH_SIZE,
    mode="train",
)

evalds = load_dataset(pattern="eval*", batch_size=1000, mode="eval").take(
    count=NUM_EVAL_EXAMPLES // 1000
)

steps_per_epoch = NUM_TRAIN_EXAMPLES // (TRAIN_BATCH_SIZE * NUM_EVALS)

logdir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callback = keras.callbacks.TensorBoard(
    log_dir=logdir, histogram_freq=1
)

history = model.fit(
    trainds,
    validation_data=evalds,
    epochs=NUM_EVALS,
    steps_per_epoch=steps_per_epoch,
    callbacks=[tensorboard_callback],
)

### 損失曲線の可視化


In [ ]:
# プロット
nrows = 1
ncols = 2
fig = plt.figure(figsize=(10, 5))

for idx, key in enumerate(["loss", "root_mean_squared_error"]):
    ax = fig.add_subplot(nrows, ncols, idx + 1)
    plt.plot(history.history[key])
    plt.plot(history.history[f"val_{key}"])
    plt.title(f"model {key}")
    plt.ylabel(key)
    plt.xlabel("epoch")
    plt.legend(["train", "validation"], loc="upper left");

### モデルの保存


In [ ]:
OUTPUT_DIR = "babyweight_trained_wd"
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

EXPORT_PATH = os.path.join(OUTPUT_DIR, TIMESTAMP)

SAVEDMODEL_PATH = os.path.join(EXPORT_PATH, "savedmodel")
KERAS_PATH = os.path.join(EXPORT_PATH, "model.keras")

model.export(EXPORT_PATH)
model.save(KERAS_PATH)

In [ ]:
!ls $EXPORT_PATH

## ラボのまとめ
このラボでは、まずデータ入力用の CSV カラム名、ラベルカラム、およびカラムのデフォルト値を定義しました。次に、CSV ファイルから特徴量とラベルで構成される `tf.data Dataset` を構築し、生の特徴量に対する入力レイヤーを作成しました。続いて、モデル入力用の前処理レイヤーをセットアップし、Keras で Wide & Deep ニューラルネットワークを構築しました。カスタム評価指標を作成して Wide & Deep モデルを構築し、最後にモデルの訓練と評価を行いました。


Copyright 2026 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.